<a href="https://colab.research.google.com/github/TYMohamedWael/MOD/blob/master/nllb_200_distilled_600M.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/facebook/nllb-200-distilled-600M

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/facebook/nllb-200-distilled-600M)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [1]:
# Use a pipeline as a high-level helper
# Warning: Pipeline type "translation" is no longer supported in transformers v5.
# You must load the model directly (see below) or downgrade to v4.x with:
# 'pip install "transformers<5.0.0'

pip install "transformers<5.0.0
from transformers import pipeline

pipe = pipeline("translation", model="facebook/nllb-200-distilled-600M")

SyntaxError: unterminated string literal (detected at line 6) (1332700098.py, line 6)

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")

In [2]:
!pip install -q transformers sentencepiece torch accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.6.0
    Uninstalling huggingface_hub-1.6.0:
      Successfully uninstalled huggingface_hub-1.6.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [5]:
pip install transformers torch sentencepiece python-docx fastapi uvicorn tqdm

In [3]:
from transformers import pipeline

pipe = pipeline(
    "translation",
    model="facebook/nllb-200-distilled-600M",
    src_lang="eng_Latn",
    tgt_lang="arb_Arab"
)

text = "Hello, how are you?"

result = pipe(text)

print(result[0]["translation_text"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


مرحباً، كيف حالك؟


In [10]:
import os
import torch
import re
from tqdm import tqdm
from docx import Document
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import time

# =========================
# MODEL CONFIG
# =========================
MODEL_NAME = "facebook/nllb-200-distilled-600M"
SRC_LANG = "arb_Arab"
TGT_LANG = "hau_Latn"

device = "cuda" if torch.cuda.is_available() else "cpu"
translation_cache = {}  # Simple Translation Memory

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

def translate_text(text):
    """Translates a single string with caching and sentence splitting."""
    if not text.strip():
        return text

    if text in translation_cache:
        return translation_cache[text]

    # Split into sentences for better NLLB performance
    sentences = re.split(r'([.؟!?])', text)
    translated_parts = []

    for i in range(0, len(sentences)-1, 2):
        sentence = sentences[i] + (sentences[i+1] if i+1 < len(sentences) else "")
        if not sentence.strip(): continue

        inputs = tokenizer(sentence, return_tensors="pt").to(device)
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(TGT_LANG),
            max_length=512
        )
        translated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        translated_parts.append(translated)

    # If no splitting occurred or regex failed to find punctuation
    if not translated_parts:
        inputs = tokenizer(text, return_tensors="pt").to(device)
        outputs = model.generate(**inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids(TGT_LANG))
        final_translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    else:
        final_translation = " ".join(translated_parts)

    translation_cache[text] = final_translation
    return final_translation

def translate_docx(input_file, output_file):
    print(f"Processing: {input_file}")
    doc = Document(input_file)

    # Total paragraphs for progress bar
    for paragraph in tqdm(doc.paragraphs, desc="Translating Paragraphs"):
        if paragraph.text.strip():
            # We iterate through 'runs' to preserve formatting (Bold/Italic)
            for run in paragraph.runs:
                if run.text.strip():
                    run.text = translate_text(run.text)

    doc.save(output_file)
    print(f"Saved to: {output_file}")

if __name__ == "__main__":
    # Ensure the input file exists before running
    if os.path.exists("book.docx"):
        translate_docx("book.docx", "book_hausa_improved.docx")
    else:
        print("Error: book.docx not found. Please upload it to the content folder.")

Loading model...
Processing: book.docx


Translating Paragraphs: 100%|██████████| 15/15 [00:53<00:00,  3.57s/it]

Saved to: book_hausa_improved.docx
